# Exploratory Data Analysis — UCI Drug Consumption Dataset (Raw)

This notebook explores the raw UCI dataset before encoding, examining feature distributions, class balance across drug types, and relationships between personality traits and drug use.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

ALL_COLS = [
    'ID', 'Age', 'Gender', 'Education', 'Country', 'Ethnicity',
    'Nscore', 'Escore', 'Oscore', 'Ascore', 'Cscore', 'Impulsive', 'SS',
    'Alcohol', 'Amphet', 'Amyl', 'Benzos', 'Caff', 'Cannabis', 'Choc',
    'Coke', 'Crack', 'Ecstasy', 'Heroin', 'Ketamine', 'Legalh', 'LSD',
    'Meth', 'Mushrooms', 'Nicotine', 'Semer', 'VSA'
]

df = pd.read_csv('drug_consumption.data', header=None, names=ALL_COLS)

print(df.shape)
display(df.head())
print(df.columns.tolist())
print(df.isnull().sum())
print(df.describe(include='all'))

In [ ]:
# Cannabis class distribution — our chosen target variable
sns.countplot(x='Cannabis', data=df, order=['CL0','CL1','CL2','CL3','CL4','CL5','CL6'],
              palette='Blues_d')
plt.title('Cannabis Usage Class Distribution')
plt.xlabel('Usage Class (CL0=Never, CL6=Daily)')
plt.ylabel('Count')
plt.show()

print(df['Cannabis'].value_counts().sort_index())

In [ ]:
# Missing values heatmap
sns.heatmap(df.isnull(), cbar=False, yticklabels=False)
plt.title('Missing Values Heatmap')
plt.show()

In [ ]:
# Distribution of all 12 personality/demographic features
FEATURE_COLS = ['Age','Gender','Education','Country','Ethnicity',
                'Nscore','Escore','Oscore','Ascore','Cscore','Impulsive','SS']
FEATURE_LABELS = ['Age','Gender','Education','Country','Ethnicity',
                  'Neuroticism','Extraversion','Openness',
                  'Agreeableness','Conscientiousness','Impulsivity','Sensation-Seeking']

fig, axes = plt.subplots(3, 4, figsize=(16, 10))
axes = axes.flatten()
for i, (col, label) in enumerate(zip(FEATURE_COLS, FEATURE_LABELS)):
    axes[i].hist(df[col], bins=20, color='steelblue', edgecolor='white', alpha=0.85)
    axes[i].set_title(label, fontsize=10)
    axes[i].set_xlabel('Quantified Value')
    axes[i].set_ylabel('Count')
fig.suptitle('Feature Distributions — UCI Drug Consumption Dataset', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Usage rate across all 18 drugs (% of respondents who have ever used)
DRUG_COLS = ['Alcohol','Amphet','Amyl','Benzos','Caff','Cannabis','Choc',
             'Coke','Crack','Ecstasy','Heroin','Ketamine','Legalh','LSD',
             'Meth','Mushrooms','Nicotine','Semer','VSA']

ever_used = {d: (df[d] != 'CL0').mean() * 100 for d in DRUG_COLS}
ever_used_s = pd.Series(ever_used).sort_values(ascending=True)

ever_used_s.plot(kind='barh', figsize=(10, 7), color='steelblue', alpha=0.85)
plt.xlabel('% of Respondents Who Have Ever Used')
plt.title('Drug Usage Prevalence Across All Substances')
plt.tight_layout()
plt.show()

In [ ]:
# Gender distribution
sns.countplot(x='Gender', data=df)
plt.title('Gender Distribution')
plt.xticks([0, 1], ['Male (-0.48)', 'Female (0.48)'])
plt.show()

print(df['Gender'].value_counts())

In [ ]:
# Correlation heatmap of personality features
corr = df[FEATURE_COLS].corr()
plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm',
            xticklabels=FEATURE_LABELS, yticklabels=FEATURE_LABELS)
plt.title('Personality Feature Correlation Matrix')
plt.tight_layout()
plt.show()

In [ ]:
# Binary cannabis target vs personality scores (boxplots)
df['cannabis_binary'] = df['Cannabis'].apply(lambda x: 0 if x in ['CL0','CL1'] else 1)

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()
personality_cols = ['Nscore','Escore','Oscore','Ascore','Cscore','Impulsive','SS']
personality_labels = ['Neuroticism','Extraversion','Openness','Agreeableness',
                      'Conscientiousness','Impulsivity','Sensation-Seeking']

for i, (col, label) in enumerate(zip(personality_cols, personality_labels)):
    df.boxplot(column=col, by='cannabis_binary', ax=axes[i])
    axes[i].set_title(label)
    axes[i].set_xlabel('0=Non-user  1=At Risk')
    axes[i].set_ylabel('Score')

fig.suptitle('Personality Scores by Cannabis Risk Class', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()